# Debug VWAP → bybit_2 (no trades despite signals)

**Ad-hoc operator runbook.**

Use this when the liveness watchdog reports actionable VWAP signals firing
but no trades landing on `bybit_2` — the pattern that has now recurred at
BUG-045 / BUG-046 / BUG-048 / BUG-049 / today's #441 + #446.

## What this notebook does

1. SSHes to the VM (read-only first).
2. **Reads** the live state and prints exactly which gate is silencing VWAP:
   - The four `.env` flags that control the dispatch path
   - All open `order_packages` for `vwap` (linked vs unlinked)
   - All open `trades` rows for `vwap` on `bybit_2`
   - bybit_2's actual free BTC + USDT balances on Bybit V5 UNIFIED
   - Last 30 lines of `signal_audit.jsonl` filtered to `vwap` (the
     `status` + `reason` field per tick is the smoking gun)
   - Last 50 WARN/ERROR lines from `journalctl -u ict-trader-live`
     mentioning vwap / bybit_2
3. Prints a **decision matrix** mapping what you saw to which action to take.
4. **Action 1 (cell 5)** — clear a stuck strategy-monocle gate by orphaning
   `vwap` open packages and their linked open trade rows. Gated by
   `CONFIRM = True`. Reversible.

## What this notebook does NOT do

- It does not modify code, configs, secrets, or the trader systemd unit.
- It does not place or cancel any exchange orders.
- It does not flip the per-account `mode: live | dry_run` flag.

If the diagnosis points to a code or config fix (e.g. `MONITOR_RECONCILE_ENABLED`
missing from `.env`, or a needed change to `vwap.py`'s `monitor()` to actually
close on TP), that's a separate Claude session, not this notebook.

## Required Colab Secrets

| Name | What it holds |
|---|---|
| `VM_SSH_HOST` | VM hostname or public IP |
| `VM_SSH_USER` | SSH user on the VM (usually `ubuntu`) |

## Reversibility (Action 1 only)

```sql
UPDATE order_packages SET status = 'open' WHERE status = 'orphaned'
  AND json_extract(meta, '$.orphaned_by') = 'debug_vwap_bybit2.ipynb';

UPDATE trades SET status = 'open', exit_reason = NULL
  WHERE status = 'orphaned'
  AND json_extract(notes, '$.orphaned_by') = 'debug_vwap_bybit2.ipynb';
```

## Pre-filled VM constants

Mirrors `docs/claude/operating-protocol.md` § 7. The notebook reads
`VM_SSH_HOST` and `VM_SSH_USER` from Colab Secrets so you can run it
unmodified.

In [ ]:
# Cell 1: SSH setup. Mirrors notebooks/operator/sweep_unlinked_packages.ipynb.
import os, shutil, stat, subprocess, tempfile
from google.colab import drive, userdata

print("⏳ Mounting Google Drive…")
try:
    drive.mount("/content/drive")
except Exception as exc:
    print(f"⚠️  drive.mount() raised: {exc}")

DEFAULT_SSH_KEY_NAMES = [
    "ict-bot-ovm-private.key",
    "vm_ssh_key", "id_rsa", "id_ed25519", "id_ecdsa",
]
DRIVE_FOLDER = "/content/drive/MyDrive/ICT_Bot_Secrets"

ssh_key_path = None
if os.path.exists("/content/drive/MyDrive"):
    for name in DEFAULT_SSH_KEY_NAMES:
        path = os.path.join(DRIVE_FOLDER, name)
        if os.path.exists(path):
            ssh_key_path = path
            break
if ssh_key_path is None:
    raise SystemExit(
        f"No SSH key found in {DRIVE_FOLDER}/. Place "
        "`ict-bot-ovm-private.key` there and re-run."
    )
print(f"✅ SSH key located: {ssh_key_path}")

host = userdata.get("VM_SSH_HOST")
user = userdata.get("VM_SSH_USER")
if not host or not user:
    raise SystemExit("VM_SSH_HOST + VM_SSH_USER must be set in Colab Secrets.")

tmpdir = tempfile.mkdtemp(prefix="debug-vwap-bybit2-")
key_path = os.path.join(tmpdir, "vm_key")
shutil.copy(ssh_key_path, key_path)
os.chmod(key_path, stat.S_IRUSR | stat.S_IWUSR)

ssh_target = f"{user}@{host}"
ssh_opts = [
    "-i", key_path,
    "-o", "StrictHostKeyChecking=accept-new",
    "-o", "UserKnownHostsFile=" + os.path.join(tmpdir, "known_hosts"),
    "-o", "ConnectTimeout=15",
    "-o", "BatchMode=yes",
]

def run_ssh(cmd, *, label, allow_fail=False):
    proc = subprocess.run(
        ["ssh"] + ssh_opts + [ssh_target, cmd],
        capture_output=True, text=True,
    )
    if proc.returncode != 0 and not allow_fail:
        print(f"❌ {label} failed (exit {proc.returncode})")
        print((proc.stderr or proc.stdout)[:500])
        raise SystemExit(f"{label} failed")
    return (proc.stdout or "").strip()

print(f"⏳ Connecting to {ssh_target} …")
run_ssh("echo connected", label="SSH connectivity")
print("✅ SSH OK")

REPO_DIR = f"/home/{user}/ict-trading-bot"
DB_PATH = f"{REPO_DIR}/trade_journal.db"
ENV_PATH = f"{REPO_DIR}/.env"
AUDIT_PATH = f"{REPO_DIR}/runtime_logs/signal_audit.jsonl"


In [ ]:
# Cell 2: READ-ONLY DIAGNOSTIC.
# Pulls everything needed to identify which gate is silencing VWAP on bybit_2.
# No DB writes, no exchange calls.

print("=" * 72)
print("PART 1 — .env flags that gate the VWAP→bybit_2 path")
print("=" * 72)
# `.env` is mode 600 owned by ubuntu and not readable as another user;
# we are sshed as ubuntu so cat is fine. We grep only the four flags
# that affect this path so the rest of the file (secrets!) never leaves
# the VM.
env_flags = run_ssh(
    f"grep -E '^(MULTI_ACCOUNT_DISPATCH|MONITOR_RECONCILE_ENABLED|"
    f"MONITOR_APPLY_TO_EXCHANGE|STRATEGY|BYBIT_TESTNET)=' {ENV_PATH} "
    f"|| true",
    label="env flag grep", allow_fail=True,
)
print(env_flags or "(none of the four flags set — all on code defaults)")
print()
# Presence-only check for the bybit_2 API key. Never echo the value.
key_check = run_ssh(
    f"grep -c '^BYBIT_API_KEY_2=' {ENV_PATH} || true",
    label="BYBIT_API_KEY_2 presence",
)
print(f"BYBIT_API_KEY_2 present in .env: {'yes' if key_check.strip() not in ('0','') else 'NO — bybit_2 cannot live-trade'}")
secret_check = run_ssh(
    f"grep -c '^BYBIT_API_SECRET_2=' {ENV_PATH} || true",
    label="BYBIT_API_SECRET_2 presence",
)
print(f"BYBIT_API_SECRET_2 present in .env: {'yes' if secret_check.strip() not in ('0','') else 'NO — bybit_2 cannot live-trade'}")
print()

print("=" * 72)
print("PART 2 — open VWAP order_packages (the strategy-monocle gate query)")
print("=" * 72)
# Linked rows are what _has_open_package_for_strategy('vwap',
# linked_only=True) sees — if any exist, every VWAP signal returns
# status=skipped, reason=open_package_exists.
linked_sql = (
    "SELECT order_package_id, symbol, direction, entry, sl, tp, "
    "status, linked_trade_id, datetime(created_at) AS created_at "
    "FROM order_packages "
    "WHERE strategy_name='vwap' AND status='open' AND linked_trade_id IS NOT NULL "
    "ORDER BY datetime(created_at) DESC;"
)
linked = run_ssh(
    f'sqlite3 -header -column {DB_PATH} "{linked_sql}"',
    label="linked open vwap packages",
)
print("LINKED open VWAP packages (these silence the strategy-monocle gate):")
print(linked or "(none — monocle gate is NOT the cause)")
print()
# Unlinked rows: BUG-049 fix means these no longer block the gate, but
# they're worth seeing because they tell us how often dispatch is
# rejecting before placement.
unlinked_sql = (
    "SELECT order_package_id, symbol, direction, status, "
    "datetime(created_at) AS created_at "
    "FROM order_packages "
    "WHERE strategy_name='vwap' AND status='open' AND linked_trade_id IS NULL "
    "ORDER BY datetime(created_at) DESC LIMIT 10;"
)
unlinked = run_ssh(
    f'sqlite3 -header -column {DB_PATH} "{unlinked_sql}"',
    label="unlinked open vwap packages",
)
print("UNLINKED open VWAP packages (informational — do NOT block the gate post-BUG-049):")
print(unlinked or "(none)")
print()

print("=" * 72)
print("PART 3 — open trades for vwap on bybit_2")
print("=" * 72)
trades_sql = (
    "SELECT id, symbol, direction, position_size, entry_price, "
    "status, exit_reason, datetime(created_at) AS created_at "
    "FROM trades "
    "WHERE strategy_name='vwap' AND account_id='bybit_2' "
    "AND status='open' AND COALESCE(is_backtest,0)=0 "
    "ORDER BY datetime(created_at) DESC;"
)
open_trades = run_ssh(
    f'sqlite3 -header -column {DB_PATH} "{trades_sql}"',
    label="open vwap bybit_2 trades",
)
print("Open VWAP trades on bybit_2 (DB view):")
print(open_trades or "(none)")
print()

print("=" * 72)
print("PART 4 — bybit_2 actual free balances on Bybit V5 UNIFIED")
print("=" * 72)
# Re-uses _fetch_spot_coin_balances() so we get the exact same numbers
# the spot-sell pre-flight guard checks. Runs in the bot's venv with the
# bot's env so creds are picked up automatically.
balance_py = (
    "import os, json, sys; sys.path.insert(0, '.'); "
    "from src.units.accounts.clients import bybit_client_for; "
    "from src.units.accounts.execute import _fetch_spot_coin_balances; "
    "cfg={'account_id':'bybit_2','exchange':'bybit',"
    "'api_key_env':'BYBIT_API_KEY_2','market_type':'spot'}; "
    "client=bybit_client_for(cfg); "
    "print(json.dumps(_fetch_spot_coin_balances(client,'BTCUSDT'),indent=2)) "
    "if client else print('client=None — BYBIT_API_KEY_2/SECRET_2 missing')"
)
balances = run_ssh(
    f"cd {REPO_DIR} && /usr/bin/python3 -c \"{balance_py}\" 2>&1 | tail -20",
    label="bybit_2 balances", allow_fail=True,
)
print(balances or "(call returned nothing — see WARN below)")
print()

print("=" * 72)
print("PART 5 — last 30 vwap entries from runtime_logs/signal_audit.jsonl")
print("=" * 72)
print("(this is the smoking-gun field — status + reason per tick)")
print()
audit = run_ssh(
    f"grep -i 'vwap' {AUDIT_PATH} 2>/dev/null | tail -30 || true",
    label="signal_audit tail", allow_fail=True,
)
print(audit or "(no vwap rows in signal_audit.jsonl — the strategy may not be ticking at all)")
print()

print("=" * 72)
print("PART 6 — last 50 vwap/bybit_2 mentions in journalctl (today)")
print("=" * 72)
journal = run_ssh(
    "journalctl -u ict-trader-live --since today --no-pager "
    "| grep -iE '(vwap|bybit_2|170131|170134|170130|"
    "open_package_exists|spot sell refused|below_min_balance|"
    "account_mode_dry_run|RiskBreach)' "
    "| tail -50 || true",
    label="journalctl tail", allow_fail=True,
)
print(journal or "(nothing — either ict-trader-live isn't running or no relevant log lines today)")


---

## Decision matrix — what to do based on Cell 2's output

Read top-to-bottom; the **first** rule that matches is the one to act on.

| What Cell 2 showed | What it means | Action |
|---|---|---|
| Part 1: `BYBIT_API_KEY_2 present: NO` | bybit_2 has no exchange creds; every dispatch refuses | **Run `notebooks/operator/rotate_api_keys.ipynb`** to load the keys, then re-run this cell. Not the gate-clearing cell below. |
| Part 1: `MULTI_ACCOUNT_DISPATCH=false` | Pinned to legacy single-client path (only the first matching account ever gets the order) | Edit `.env` to `MULTI_ACCOUNT_DISPATCH=true` (or unset — default is true) and `sudo systemctl restart ict-trader-live`. Tier 2 — this changes service behavior. |
| Part 1: `MONITOR_RECONCILE_ENABLED` not present | Reconciler can never auto-orphan stuck packages — BUG-048-class hazard | Restore `MONITOR_RECONCILE_ENABLED=true` to `.env` (the runbook says it should be the default). Operator action. |
| Part 2: LINKED open VWAP packages ≥ 1 AND Part 3: open trade exists AND Part 4: balances are zero or near-zero | Strategy-monocle is locked-on — the package is `status='open'` in DB but the position is already flat at the exchange. The reconciler should have caught this. | **Cell 5 below** — this is what the notebook is for. |
| Part 2: LINKED open VWAP packages ≥ 1 AND Part 3: open trade exists AND Part 4: shows real holdings | The bot has a live position. Strategy-monocle is doing its job. Wait for SL/TP. | No action. (But note: VWAP's `monitor()` only does break-even SL — a code-level follow-up is needed for it to ever close on TP. Track that in a separate sprint, not here.) |
| Part 5 audit lines repeatedly show `reason=open_package_exists` | Same as the row above — strategy-monocle is locked-on | **Cell 5 below.** |
| Part 5 audit lines repeatedly show `Spot sell refused for BTCUSDT: insufficient free BTC balance` | bybit_2 wallet has no BTC and VWAP is firing **sell** signals (price > VWAP). The pre-flight guard refuses by design. | **Not a bug.** Sells will only execute after a buy fills. Either accept the asymmetry or fund the wallet with some BTC. Track as a strategy-design follow-up. |
| Part 5 audit lines repeatedly show `account_mode_dry_run` | Someone flipped bybit_2 to dry mode via `/accounts dry bybit_2` | Send `/accounts live bybit_2` in Telegram. No code or DB action needed. |
| Part 5 audit lines repeatedly show `below_min_balance` | bybit_2 USDT free is below `$min_balance_usd ($50)` or daily loss budget exhausted | Either fund the account or wait for next UTC midnight. No code or DB action. |
| Part 6 journalctl shows `170131 Insufficient balance` AFTER 2026-05-07 09:00 UTC | The today-#446 fix didn't deploy, or there's a third recurrence | Run `notebooks/operator/deploy_latest_main.ipynb` first, then re-run this notebook. |
| Part 6 journalctl shows `170134` | Tick-precision rejection (BUG-057 family) | Separate fix — not this notebook. |
| Cell 2 produced nothing useful (Parts 5+6 empty) | The trader may not be running at all | `sudo systemctl status ict-trader-live` from a fresh `/vm` Telegram command. |


In [ ]:
# Cell 5: ACTION — clear stuck VWAP strategy-monocle gate.
#
# ONLY run this when Cell 2's output matches the decision-matrix row that
# explicitly points here. If you are unsure: do nothing and ping Claude
# with a screenshot of Cell 2's output.
#
# Effect:
#   1. Marks every `order_packages` row WHERE strategy_name='vwap' AND
#      status='open' AND linked_trade_id IS NOT NULL  as status='orphaned'
#      with a meta breadcrumb so the change is auditable.
#   2. Cascades: every linked `trades` row also moves to status='orphaned'
#      with exit_reason='debug_vwap_bybit2_notebook' so /last5 and the
#      hourly report stop showing a phantom open position.
#
# Reversibility: see notebook header. Both UPDATEs are reversible by SQL.
#
# Effect on the trader: the strategy-monocle gate re-reads the DB on every
# tick — VWAP unblocks within one tick (no service restart).
#
# Effect on the exchange: NONE. This notebook does not place, cancel, or
# modify exchange orders. If a real position is still open on Bybit, you
# must close it manually via Telegram (`/closeall vwap` or the Bybit UI).

CONFIRM = False  # <- set to True to commit

if not CONFIRM:
    print("CONFIRM is False — no changes made.")
    print("Set CONFIRM = True and re-run this cell to commit.")
else:
    pkg_update = (
        "UPDATE order_packages "
        "SET status='orphaned', "
        "    updated_at=strftime('%Y-%m-%dT%H:%M:%SZ', 'now'), "
        "    meta=json_set(COALESCE(meta, '{}'), "
        "      '$.orphaned_at', strftime('%Y-%m-%dT%H:%M:%SZ', 'now'), "
        "      '$.orphaned_by', 'debug_vwap_bybit2.ipynb', "
        "      '$.orphaned_reason', "
        "      'strategy-monocle stuck — DB-open but exchange-flat for vwap/bybit_2') "
        "WHERE strategy_name='vwap' "
        "  AND status='open' "
        "  AND linked_trade_id IS NOT NULL;"
    )
    run_ssh(f'sqlite3 {DB_PATH} "{pkg_update}"',
            label="UPDATE order_packages")
    pkg_changed = run_ssh(
        f'sqlite3 {DB_PATH} "SELECT changes();"',
        label="package row count",
    )
    print(f"✅ order_packages: {pkg_changed} row(s) → status=\"orphaned\"")

    trade_update = (
        "UPDATE trades "
        "SET status='orphaned', "
        "    exit_reason='debug_vwap_bybit2_notebook', "
        "    notes=json_set(COALESCE(notes, '{}'), "
        "      '$.orphaned_at', strftime('%Y-%m-%dT%H:%M:%SZ', 'now'), "
        "      '$.orphaned_by', 'debug_vwap_bybit2.ipynb', "
        "      '$.orphaned_reason', "
        "      'cascaded from order_packages orphan; verify exchange flat') "
        "WHERE strategy_name='vwap' "
        "  AND account_id='bybit_2' "
        "  AND status='open' "
        "  AND COALESCE(is_backtest,0)=0;"
    )
    run_ssh(f'sqlite3 {DB_PATH} "{trade_update}"',
            label="UPDATE trades")
    trade_changed = run_ssh(
        f'sqlite3 {DB_PATH} "SELECT changes();"',
        label="trade row count",
    )
    print(f"✅ trades: {trade_changed} row(s) → status=\"orphaned\"")
    print()
    print("Verify on Telegram (no trader restart needed):")
    print("  /packages   → stuck VWAP rows should be gone")
    print("  /signals    → new VWAP signals should resume within one tick")
    print("  /last5      → phantom open trade should be gone")
    print()
    print("If a real position is still open on Bybit, close it manually:")
    print("  Telegram: /closeall vwap   (or use Bybit web UI)")
    print()
    print("Rollback:")
    print("  sqlite3 trade_journal.db \"UPDATE order_packages SET status='open'")
    print("    WHERE status='orphaned'")
    print("    AND json_extract(meta, '$.orphaned_by') = 'debug_vwap_bybit2.ipynb';\"")
    print("  sqlite3 trade_journal.db \"UPDATE trades SET status='open', exit_reason=NULL")
    print("    WHERE status='orphaned'")
    print("    AND json_extract(notes, '$.orphaned_by') = 'debug_vwap_bybit2.ipynb';\"")
    shutil.rmtree(tmpdir, ignore_errors=True)


---

## After running

1. Wait one tick (default `TICK_INTERVAL_SECONDS=900` = 15 min).
2. In Telegram:
   - `/packages` — stuck VWAP rows are gone.
   - `/signals` — a fresh VWAP row appears once price moves ≥1σ from VWAP.
   - `/last5` — the phantom open is gone; new fills appear when VWAP fires.
3. If VWAP still doesn't trade after one tick, re-run Cell 2 and check the
   decision matrix again — a second gate may now be the dominant one.

## Follow-up sprints (separate sessions — NOT this notebook)

- **VWAP `monitor()` close logic**: the v1 hook only does break-even SL.
  Spot has no exchange-side TP/SL since BUG-061, so without a strategy-side
  close path, every VWAP package on a spot account is structurally
  unclosable — the reconciler is the only thing that ever clears it.
  Tier 3 (touches strategy logic). File a sprint prompt.
- **`MONITOR_RECONCILE_ENABLED` enforcement**: the runbook says default
  is `true`, the code default is `false`, the `.env` carries the operator
  override. BUG-048 already burned us once with a render-drift drop.
  Either move the default to `true` in code, or add a startup check that
  refuses to start without it. Tier 2.
- **Strategy/wallet asymmetry**: bybit_2 holds USDT only; VWAP fires
  bidirectional signals; the spot-sell pre-flight refuses ~50% of them.
  Either constrain VWAP to one-way on USDT-only wallets, or seed the
  wallet with BTC. Tier 3 (operator preference).
